In [2]:
from google.colab import drive
drive.mount('/content/drive')
import os, json, time, numpy as np, pandas as pd
from sklearn.datasets import fetch_olivetti_faces
from sklearn.tree import DecisionTreeClassifier

OUT = "/content/drive/MyDrive/ml_a2_q3"; os.makedirs(OUT, exist_ok=True)

ROLL = "M26ECM505"
SEED = int("".join(c for c in ROLL if c.isdigit()))
K, R, B = 40, 30, 100

d = fetch_olivetti_faces(); X_all, y_all = d.data, d.target
s = np.load("split.npz"); tr, va, te = s["tr"], s["va"], s["te"]
Xtr, ytr, Xte, yte = X_all[tr], y_all[tr], X_all[te], y_all[te]
def mask(Xs):
    Xm = Xs.copy().reshape(-1, 64, 64); Xm[:, 24:40, 24:40] = 0
    return Xm.reshape(len(Xs), -1)
Xeval = np.vstack([Xte, mask(Xte)])            # first 80 rows clean, last 80 masked
N = len(Xte)

ft = json.load(open("frozen_tree.json")); fb = json.load(open("frozen_boost.json"))
msl, depth_b, T_b = ft["min_samples_leaf"], fb["depth"], fb["rounds"]
DEPTHS = [4, 16, None]
tree_full = DecisionTreeClassifier(max_depth=ft["depth"], min_samples_leaf=msl, random_state=0).fit(Xtr, ytr)
print("actual depth of the unrestricted tree:", tree_full.get_depth())

# ---- SAMME (identical to Q2) ----
def samme_round(X, y, w, depth, K, seed=0, eps_floor=1e-10):
    h = DecisionTreeClassifier(max_depth=depth, random_state=seed).fit(X, y, sample_weight=w)
    pred = h.predict(X); miss = pred != y; eps = w[miss].sum()
    if eps >= 1 - 1 / K: return h, pred, eps, None, w, "rejected"
    eps_c = max(eps, eps_floor)
    alpha = np.log((1 - eps_c) / eps_c) + np.log(K - 1)
    w_new = w * np.exp(alpha * miss); w_new /= w_new.sum()
    return h, pred, eps, alpha, w_new, "ok"

def samme_fit(X, y, depth, T, K=40):
    n = len(X); w = np.full(n, 1 / n); learners, alphas = [], []
    for t in range(T):
        h, pred, eps, alpha, w_new, status = samme_round(X, y, w, depth, K, seed=t)
        if status == "rejected": break
        learners.append(h); alphas.append(alpha); w = w_new
        if eps <= 1e-10: break
    return learners, np.array(alphas)

def samme_q(L, A, X, K=40):
    S = np.zeros((len(X), K))
    for h, a in zip(L, A): S[np.arange(len(X)), h.predict(X)] += a
    return S / A.sum()

def onehot(p):
    V = np.zeros((len(p), K)); V[np.arange(len(p)), p] = 1; return V

# ---- 30 resampled training sets, created once and saved ----
rs_file = f"{OUT}/resamples.npy"
if not os.path.exists(rs_file):
    rg = np.random.default_rng([SEED, 999]); rs = []
    for r in range(R):
        idx = []
        for c in range(K):
            own = np.where(ytr == c)[0]                 # that identity's 6 training faces
            idx += list(rg.choice(own, 6, replace=True))
        rs.append(idx)
    np.save(rs_file, np.array(rs))
rs = np.load(rs_file); print(rs.shape)                  # expect (30, 240)

Mounted at /content/drive
downloading Olivetti faces from https://ndownloader.figshare.com/files/5976027 to /root/scikit_learn_data
actual depth of the unrestricted tree: 30
(30, 240)


In [3]:
import glob
files = sorted(glob.glob(f"{OUT}/rep_*.npz"))
print("refits found:", len(files))                      # expect 30
keys = ["tree_4", "bag_4", "tree_16", "bag_16", "tree_None", "bag_None", "boost"]
P = {k: np.stack([np.load(f)[k] for f in files]).astype(np.float64) for k in keys}
# each P[k] has shape (R=30, 2 sets [0=clean, 1=masked], N=80, K=40)
E = np.eye(K)[yte]                                       # one-hot targets, (80, 40)
rounds = [int(np.load(f)["boost_rounds"]) for f in files]
print("boosting rounds used: min", min(rounds), "mean", np.mean(rounds), "max", max(rounds))

def bias_var(Pk):                                        # Pk: (R, N, K)
    pbar = Pk.mean(0)
    bias2 = ((pbar - E) ** 2).sum(1).mean()
    var = ((Pk - pbar) ** 2).sum(2).mean()
    return bias2, var

def acc_stats(Pk):
    a = (Pk.argmax(2) == yte).mean(1)                    # accuracy of each refit
    return a.mean(), a.std(ddof=1)                       # sample std over the 30 refits

rows = []
for k in keys:
    b2, v = bias_var(P[k][:, 0])                         # clean test set
    am, asd = acc_stats(P[k][:, 0])
    mm, _ = acc_stats(P[k][:, 1])                        # masked accuracy (mean)
    rows.append(dict(model=k, bias2=b2, variance=v, bias2_plus_var=b2 + v,
                     clean_acc_mean=am, clean_acc_std=asd, masked_acc_mean=mm))
dfq3 = pd.DataFrame(rows)
print(dfq3.round(4).to_string())
dfq3.to_csv("q3_results.csv", index=False)

refits found: 30
boosting rounds used: min 100 mean 100.0 max 100
       model   bias2  variance  bias2_plus_var  clean_acc_mean  clean_acc_std  masked_acc_mean
0     tree_4  1.6254    0.1729          1.7983          0.1008         0.0161           0.1008
1      bag_4  0.8886    0.0140          0.9026          0.2683         0.0568           0.2771
2    tree_16  0.9071    0.5279          1.4350          0.2825         0.0274           0.2567
3     bag_16  0.6676    0.0199          0.6875          0.6542         0.0598           0.5350
4  tree_None  0.3813    0.6212          1.0025          0.4988         0.0545           0.3362
5   bag_None  0.5193    0.0239          0.5432          0.7754         0.0312           0.6146
6      boost  0.8763    0.0168          0.8931          0.3621         0.0724           0.3325


In [4]:
rows = []
for dep in ["4", "16", "None"]:
    bt, vt = bias_var(P["tree_" + dep][:, 0])
    bb, vb = bias_var(P["bag_" + dep][:, 0])
    at, _ = acc_stats(P["tree_" + dep][:, 0])
    ab, _ = acc_stats(P["bag_" + dep][:, 0])
    rows.append(dict(depth=dep, tree_bias2=bt, tree_var=vt, bag_bias2=bb, bag_var=vb,
                     var_removed=vt - vb, var_removed_pct=100 * (vt - vb) / vt,
                     tree_acc=at, bag_acc=ab, acc_gain=ab - at))
dfdepth = pd.DataFrame(rows)
print(dfdepth.round(4).to_string())
dfdepth.to_csv("q3b_results.csv", index=False)

  depth  tree_bias2  tree_var  bag_bias2  bag_var  var_removed  var_removed_pct  tree_acc  bag_acc  acc_gain
0     4      1.6254    0.1729     0.8886   0.0140       0.1589          91.8816    0.1008   0.2683    0.1675
1    16      0.9071    0.5279     0.6676   0.0199       0.5080          96.2338    0.2825   0.6542    0.3717
2  None      0.3813    0.6212     0.5193   0.0239       0.5974          96.1607    0.4988   0.7754    0.2767


In [5]:
# Q3(c) must use leaf occupancy and disagreement, so measure them on the original training set.
from itertools import combinations
rng_c = np.random.default_rng([SEED, 4242])
rows = []
for dep in [4, 16, None]:
    t = DecisionTreeClassifier(max_depth=dep, min_samples_leaf=msl, random_state=0).fit(Xtr, ytr)
    leaf = t.tree_.children_left == -1
    sizes = t.tree_.n_node_samples[leaf]
    preds = []
    for b in range(B):
        idx = rng_c.integers(0, len(Xtr), len(Xtr))
        tb = DecisionTreeClassifier(max_depth=dep, min_samples_leaf=msl,
                                    random_state=int(rng_c.integers(1_000_000_000))).fit(Xtr[idx], ytr[idx])
        preds.append(tb.predict(Xte))
    preds = np.array(preds)
    dis = np.mean([(preds[i] != preds[j]).mean() for i, j in combinations(range(B), 2)])
    rows.append(dict(depth=str(dep), n_leaves=int(leaf.sum()), mean_per_leaf=len(Xtr) / leaf.sum(),
                     frac_singleton=(sizes == 1).mean(), disagreement_test=dis))
dfleaf = pd.DataFrame(rows)
print(dfleaf.round(3).to_string())
dfleaf.to_csv("q3c_leaf_disagreement.csv", index=False)

  depth  n_leaves  mean_per_leaf  frac_singleton  disagreement_test
0     4         5         48.000             0.0              0.948
1    16        17         14.118             0.0              0.873
2  None        41          5.854             0.0              0.794


In [6]:
clean = {k: acc_stats(P[k][:, 0])[0] for k in keys}
masked = {k: acc_stats(P[k][:, 1])[0] for k in keys}
final = []
for name, k in [("Single tree (selected)", "tree_None"), ("Bagging (B=100)", "bag_None"), ("Boosting (selected)", "boost")]:
    b2, v = bias_var(P[k][:, 0])
    final.append(dict(model=name, clean_acc=clean[k], masked_acc=masked[k], bias2=b2, variance=v))
dff = pd.DataFrame(final)
print(dff.round(4).to_string())
dff.to_csv("q3d_final_table.csv", index=False)

                    model  clean_acc  masked_acc   bias2  variance
0  Single tree (selected)     0.4988      0.3362  0.3813    0.6212
1         Bagging (B=100)     0.7754      0.6146  0.5193    0.0239
2     Boosting (selected)     0.3621      0.3325  0.8763    0.0168


In [3]:
MAX_THIS_RUN = 1          # first run: 1 refit to measure the time. Then raise to 5.

done_now = 0
for r in range(R):
    f = f"{OUT}/rep_{r:02d}.npz"
    if os.path.exists(f): continue
    if done_now >= MAX_THIS_RUN: 1
    t0 = time.time()
    Xr, yr = Xtr[rs[r]], ytr[rs[r]]
    rng = np.random.default_rng([SEED, 1000 + r])       # same numbers every time for a given r
    res = {}
    for dep in DEPTHS:
        key = str(dep)
        t = DecisionTreeClassifier(max_depth=dep, min_samples_leaf=msl, random_state=0).fit(Xr, yr)
        res["tree_" + key] = np.stack([onehot(t.predict(Xeval[:N])), onehot(t.predict(Xeval[N:]))])
        V = np.zeros((2 * N, K))
        for b in range(B):
            idx = rng.integers(0, len(Xr), len(Xr))
            tb = DecisionTreeClassifier(max_depth=dep, min_samples_leaf=msl,
                                        random_state=int(rng.integers(1_000_000_000))).fit(Xr[idx], yr[idx])
            V[np.arange(2 * N), tb.predict(Xeval)] += 1
        V /= B
        res["bag_" + key] = np.stack([V[:N], V[N:]])
    L, A = samme_fit(Xr, yr, depth_b, T_b)
    q = samme_q(L, A, Xeval)
    res["boost"] = np.stack([q[:N], q[N:]])
    res["boost_rounds"] = np.array(len(L))
    np.savez_compressed(f, **{k: v.astype(np.float32) for k, v in res.items()})
    done_now += 1
    acc = {k: round(float((v[0].argmax(1) == yte).mean()), 3) for k, v in res.items() if k != "boost_rounds"}
    print(f"rep {r} done in {time.time()-t0:.0f}s | rounds used by boosting: {len(L)} | clean acc: {acc}")

print("finished reps:", len([x for x in os.listdir(OUT) if x.startswith("rep_")]), "of", R)

rep 0 done in 450s | rounds used by boosting: 100 | clean acc: {'tree_4': 0.087, 'bag_4': 0.287, 'tree_16': 0.225, 'bag_16': 0.675, 'tree_None': 0.4, 'bag_None': 0.787, 'boost': 0.463}
rep 1 done in 455s | rounds used by boosting: 100 | clean acc: {'tree_4': 0.062, 'bag_4': 0.275, 'tree_16': 0.225, 'bag_16': 0.625, 'tree_None': 0.487, 'bag_None': 0.738, 'boost': 0.25}
rep 2 done in 442s | rounds used by boosting: 100 | clean acc: {'tree_4': 0.1, 'bag_4': 0.3, 'tree_16': 0.3, 'bag_16': 0.675, 'tree_None': 0.512, 'bag_None': 0.812, 'boost': 0.45}
rep 3 done in 438s | rounds used by boosting: 100 | clean acc: {'tree_4': 0.125, 'bag_4': 0.237, 'tree_16': 0.287, 'bag_16': 0.637, 'tree_None': 0.575, 'bag_None': 0.738, 'boost': 0.275}
rep 4 done in 432s | rounds used by boosting: 100 | clean acc: {'tree_4': 0.1, 'bag_4': 0.338, 'tree_16': 0.3, 'bag_16': 0.787, 'tree_None': 0.487, 'bag_None': 0.8, 'boost': 0.287}
rep 5 done in 438s | rounds used by boosting: 100 | clean acc: {'tree_4': 0.1, 'b